In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import copy
import joblib
import json
import random
from scipy.stats import spearmanr
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

# 设置所有随机种子（确保实验可重复）
SEED = 42  # 可以换成您喜欢的任何种子值
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # 如果使用多GPU
torch.backends.cudnn.deterministic = True  # 保证卷积结果也是确定的
torch.backends.cudnn.benchmark = False  # 关闭基准优化，保证可重复性


/home/ma-user/anaconda3/envs/PyTorch-1.10.2/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def split_ml_data(X,y,test_size = 0.2,batch_size = 1024):
    '''
    用于确定性模型的分割参照本函数
    '''
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    # 转换为Tensor（在CPU上创建）
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), 
                                 torch.tensor(y_train, dtype=torch.float32))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), 
                                torch.tensor(y_test, dtype=torch.float32))
    # 创建DataLoader，去掉pin_memory
    batch_size = 1024
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    return train_loader,test_loader

def split_time_series(dataset,train_ratio=0.8,val_ratio=0.2,batch_size=64):
    # Split into train, validation, and test sets (e.g., 80-10-10)
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader,val_loader,test_loader

input_window=96

class TimeSeriesDataset(Dataset):
    def __init__(self, covariates, target, input_window=input_window, pred_window=24, stride=1):
        self.covariates = torch.tensor(covariates.values, dtype=torch.float32)
        self.target = torch.tensor(target.values, dtype=torch.float32)
        self.input_window = input_window
        self.pred_window = pred_window
        self.stride = stride
        self.num_samples = (len(covariates) - input_window - pred_window + 1) // stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_idx = idx * self.stride
        x = self.covariates[start_idx:start_idx + self.input_window]
        y = self.target[start_idx + self.input_window:start_idx + self.input_window + self.pred_window]
        return x, y

In [3]:
#1、数据读取
data  = pd.read_csv('train.csv').iloc[:,1:]

#2、统计量
stats = data.agg(['max', 'min']).T
max1 = stats['max']
min1 = stats['min']

stats.to_csv('stats.csv')

#3、向量化归一化
data_normalized = (data - min1) / (max1 - min1)
data_normalized = data_normalized.dropna(axis=1, how='all')

#超前对齐(仅用于ml)
data_normalized['target_123_ml'] = data_normalized['信号123'].shift(-24)
data_normalized['target_124_ml'] = data_normalized['信号124'].shift(-24)



In [4]:
#4、选用的特征如下
selected_123 = pd.read_csv('selected_cols_123.csv')['Column_Names'].values
selected_124 = pd.read_csv('selected_cols_124.csv')['Column_Names'].values



In [5]:
selected_123 = np.array(['信号42', '信号43', '信号73', '信号76', '信号123'], dtype=object)
selected_124 = np.array(['信号2', '信号6', '信号16', '信号22', '信号23', '信号24', '信号25', '信号26', '信号27', '信号28', '信号50', '信号53', '信号56', '信号60', '信号61', '信号63', '信号64', '信号77', '信号86', '信号96', '信号104', '信号106', '信号119', '信号124'], dtype=object)

In [6]:
#5、确保使用GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.set_per_process_memory_fraction(0.8)

#6机器学习使用的数据集
X_123 = data_normalized[selected_123].values.astype(np.float32)
y_123 = data_normalized['target_123_ml'].values.astype(np.float32)

X_124 = data_normalized[selected_124].values.astype(np.float32)
y_124 = data_normalized['target_124_ml'].values.astype(np.float32)

#使用下列几个loader训练确定性机器学习模型
train_loader_123,test_loader_123 = split_ml_data(X_123,y_123)
train_loader_124,test_loader_124 = split_ml_data(X_124,y_124)

#7、时间序列模型是用的数据集：LSTM、TCN、transformer
#这里输入窗口长度为48，输出24
input_window_len = input_window
output_window_len = 24

dataset_123 = TimeSeriesDataset(data_normalized[selected_123], data_normalized[['信号123']], input_window=input_window_len, pred_window=output_window_len, stride=24)
dataset_124 = TimeSeriesDataset(data_normalized[selected_124], data_normalized[['信号124']], input_window=input_window_len, pred_window=output_window_len, stride=24)

train_loader_ts_123,val_loader_ts_123,test_loader_ts_123 = split_time_series(dataset_123)
train_loader_ts_124,val_loader_ts_124,test_loader_ts_124 = split_time_series(dataset_124)

In [7]:
#算法模型的接口参照此处

def KalMan(input_KALMAN):
    '''
    这里我乱写的虚构的方法
    input_KALMAN是array，shape=（24,4），表示24个时间步的4个模型预测结果
    '''
    ratio = np.random.random(4)
    return (ratio *input_KALMAN).sum(axis=1)

class NeuralNetwork(nn.Module):
    '''
    机器学习模型的标准接口形式
    '''
    def __init__(self, input_dim):
        super(NeuralNetwork, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.model(x)

class TimeSeriesTransformer(nn.Module):
    '''
    时间序列模型的标准接口形式
    其中pred_window=48
    '''
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, pred_window, nhead=4):
        super(TimeSeriesTransformer, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim*4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim * pred_window)

    def forward(self, x):
        x = self.input_projection(x)  # [batch_size, seq_len, input_dim] -> [batch_size, seq_len, hidden_dim]
        out = self.transformer(x)  # [batch_size, seq_len, hidden_dim]
        out = self.fc(out[:, -1, :])  # Use last output
        return out.view(x.size(0), -1, 1)  # [batch_size, pred_window, 1]
     

'''以下是两类模型的初始化'''
model = NeuralNetwork(5)
# Model parameters
input_dim_123 = 5  # From i.shape [64, 48, 5]
hidden_dim = 64
num_layers = 2
output_dim = 1  # Univariate target
pred_window = 24
nhead = 4  # Number of attention heads

# Initialize model
model_123 = TimeSeriesTransformer(input_dim_123, hidden_dim, num_layers, output_dim, pred_window, nhead)


- 给定固定格式输入情况下，输出的情况

- 开始输入信号

In [8]:
input_data = pd.read_csv('train.csv').iloc[100:155,1:]
#输入信号是全的

In [9]:
selected_123 = pd.read_csv('selected_cols_123.csv')['Column_Names'].values
index_123 = [int(i[2:]) for i in selected_123]
selected_124 = pd.read_csv('selected_cols_124.csv')['Column_Names'].values
index_124 = [int(i[2:]) for i in selected_124]

data_123 = input_data[selected_123]
data_124 = input_data[selected_124]

max_use_123 = pd.read_csv('stats.csv')[['max']].loc[index_123]
min_use_123 = pd.read_csv('stats.csv')[['min']].loc[index_123]
max_use_124 = pd.read_csv('stats.csv')[['max']].loc[index_124]
min_use_124 = pd.read_csv('stats.csv')[['min']].loc[index_124]

data_123_nor = (data_123 - min_use_123.values.T)/(max_use_123.values - min_use_123.values).T
data_124_nor = (data_124 - min_use_124.values.T)/(max_use_124.values - min_use_124.values).T

input_ml_123 = data_123_nor.iloc[-24:,:].values
input_ml_124 = data_124_nor.iloc[-24:,:].values

input_window = 48
input_ts_123 = data_123_nor.iloc[-1*input_window:,:].values.reshape(1,input_window,-1)#匹配窗口长度
input_ts_124 = data_124_nor.iloc[-1*input_window:,:].values.reshape(1,input_window,-1)#匹配窗口长度

In [10]:
'''以信号123为例'''

model = NeuralNetwork(5)#读取最佳模型，这里用初始化的代替
output_123 = model(torch.tensor(input_ml_123, dtype=torch.float32)).squeeze().detach().numpy()

'''以下是3个时间序列模型'''
model_1 = TimeSeriesTransformer(input_dim_123, hidden_dim, num_layers, output_dim, pred_window, nhead)
#读取最佳模型，这里用初始化的代替
output_123_ts1 = model_1(torch.tensor(input_ts_123, dtype=torch.float32)).squeeze().detach().numpy()

model_2 = TimeSeriesTransformer(input_dim_123, hidden_dim, num_layers, output_dim, pred_window, nhead)
#读取最佳模型，这里用初始化的代替
output_123_ts2 = model_2(torch.tensor(input_ts_123, dtype=torch.float32)).squeeze().detach().numpy()

model_3 = TimeSeriesTransformer(input_dim_123, hidden_dim, num_layers, output_dim, pred_window, nhead)
#读取最佳模型，这里用初始化的代替
output_123_ts3 = model_3(torch.tensor(input_ts_123, dtype=torch.float32)).squeeze().detach().numpy()

input_KALMAN = np.stack([output_123,output_123_ts1,output_123_ts2,output_123_ts3]).T
output_KALMAN = KalMan(input_KALMAN)

pre_signal_123 = output_KALMAN * (max_use_123.loc[123].values[0] - min_use_123.loc[123].values[0]) + min_use_123.loc[123].values[0]

- 得到未来24步的输出预测

In [11]:
pre_signal_123

array([-0.25861156,  1.07160036, -0.22809703,  0.19183773, -2.28656091,
       -0.66520497,  2.9980498 ,  0.56950927,  1.60294619,  1.62830332,
       -2.13457753, -0.96373801,  1.51258906, -1.56785663,  0.00367979,
        2.35561697,  2.2393727 , -0.15503743, -0.73736736,  1.00641755,
       -0.10984903, -0.62015377, -1.09355236, -0.71145331])

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.cuda.amp as amp
import numpy as np
import copy
import json
import os
import gc
import sys
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# 设置输出文件夹
output_dir = 'lsM16'

# 环境配置和显存清理
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"
print("CUDA可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU名称:", torch.cuda.get_device_name(0))
    print("GPU总内存:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
    torch.cuda.empty_cache()
    gc.collect()

# 使用接口定义的 input_window
input_window_len = input_window if 'input_window' in globals() else 96  # 默认 96
output_window_len = 24

# 通用 TimeSeriesDataset（如果接口未提供）
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, input_window, pred_window, stride=1):
        self.X = X.values if isinstance(X, pd.DataFrame) else X
        self.y = y.values if isinstance(y, pd.DataFrame) else y
        self.input_window = input_window
        self.pred_window = pred_window
        self.stride = stride
        self.indices = list(range(0, len(self.X) - input_window - pred_window + 1, stride))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        X = self.X[i:i + self.input_window]
        y = self.y[i + self.input_window:i + self.input_window + self.pred_window]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# 数据增强（轻微噪声）
class TimeSeriesDatasetWithNoise(Dataset):
    def __init__(self, dataset, noise_std=0.01):
        self.dataset = dataset
        self.noise_std = noise_std
        self.training = False

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.training and self.noise_std > 0:
            noise = torch.normal(0, self.noise_std, size=x.shape)
            x = x + noise
        return x, y

# 信号 123 的 LSTM+GRU 模型
class LSTMGRUForecaster(nn.Module):
    def __init__(self, input_dim, lstm_hidden_dim, gru_hidden_dim, lstm_layers, gru_layers, output_dim, pred_window, dropout=0.15):
        super(LSTMGRUForecaster, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=lstm_hidden_dim, num_layers=lstm_layers, 
                           batch_first=True, dropout=dropout if lstm_layers > 1 else 0.0)
        self.gru = nn.GRU(input_size=lstm_hidden_dim, hidden_size=gru_hidden_dim, num_layers=gru_layers, 
                         batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.residual_fc = nn.Linear(input_dim, gru_hidden_dim)
        self.fc = nn.Linear(gru_hidden_dim, pred_window * output_dim)
        self.pred_window = pred_window
        self.output_dim = output_dim

    def forward(self, x):
        residual = self.residual_fc(x[:, -1, :])
        lstm_out, _ = self.lstm(x)
        gru_out, _ = self.gru(lstm_out)
        last_hidden = gru_out[:, -1, :] + residual
        out = self.fc(last_hidden)
        out = out.view(x.size(0), self.pred_window, self.output_dim)
        return out

# 信号 124 的 CNN+LSTM 模型
class CNNLSTMForecaster(nn.Module):
    def __init__(self, input_dim, lstm_hidden_dim, num_layers, output_dim, pred_window, dropout=0.2):
        super(CNNLSTMForecaster, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.conv4 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.bn4 = nn.BatchNorm1d(128)
        self.lstm = nn.LSTM(input_size=128, hidden_size=lstm_hidden_dim, num_layers=num_layers, 
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.residual_fc = nn.Linear(input_dim, lstm_hidden_dim)
        self.fc = nn.Linear(lstm_hidden_dim, pred_window * output_dim)
        self.pred_window = pred_window
        self.output_dim = output_dim

    def forward(self, x):
        residual = self.residual_fc(x[:, -1, :])
        x = x.transpose(1, 2)  # (batch, input_dim, seq_len)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = x.transpose(1, 2)  # (batch, seq_len, 128)
        lstm_out, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1] + residual
        out = self.fc(last_hidden)
        out = out.view(x.size(0), self.pred_window, self.output_dim)
        return out

# 学习率 warmup 调度器
class WarmupLR(optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, base_lr, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.base_lr = base_lr
        super(WarmupLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            return [self.base_lr * (self.last_epoch + 1) / self.warmup_epochs for _ in self.optimizer.param_groups]
        return [self.base_lr for _ in self.optimizer.param_groups]

# 训练函数
def train_ts_model(model, train_loader, val_loader, device, model_name, epochs=150, patience=80):
    model = model.to(device)
    criterion = nn.HuberLoss(delta=0.5)
    base_lr = 0.00008 if '123' in model_name else 0.00007
    optimizer = optim.AdamW(model.parameters(), lr=base_lr, weight_decay=3e-3)
    warmup_scheduler = WarmupLR(optimizer, warmup_epochs=15, base_lr=base_lr)
    plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)
    scaler = amp.GradScaler()

    best_mse = float('inf')
    counter = 0
    best_model_state = None
    best_metrics = {}
    loss_history = {'epoch': [], 'train_loss': [], 'val_mse': []}

    for epoch in range(epochs):
        model.train()
        train_loader.dataset.training = True
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            optimizer.zero_grad()
            with amp.autocast():
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        train_loader.dataset.training = False
        total_sse = 0.0
        total_num = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)
                outputs = model(X_batch)
                sse = F.mse_loss(outputs, y_batch, reduction='sum').item()
                num_elements = outputs.numel()
                total_sse += sse
                total_num += num_elements
        val_mse = total_sse / total_num if total_num > 0 else float('inf')

        loss_history['epoch'].append(epoch + 1)
        loss_history['train_loss'].append(train_loss)
        loss_history['val_mse'].append(val_mse)
        print(f'轮次 {epoch+1}/{epochs}, 训练损失: {train_loss:.6f}, 验证MSE: {val_mse:.6f}')

        if val_mse < best_mse:
            best_mse = val_mse
            best_model_state = copy.deepcopy(model.state_dict())
            best_metrics = {'val_mse': val_mse, 'epoch': epoch + 1}
            model_path = os.path.join(output_dir, 'best_' + model_name + '.pth')
            metrics_path = os.path.join(output_dir, 'best_metrics_' + model_name + '.json')
            torch.save(best_model_state, model_path)
            with open(metrics_path, 'w') as f:
                json.dump(best_metrics, f, indent=4)
            print(f'保存最佳模型: {model_path} (验证MSE: {best_mse:.6f})')
            print(f'保存最佳指标: {metrics_path}')
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f'在轮次 {epoch+1} 提前停止')
                break

        if epoch < 15:
            warmup_scheduler.step()
        else:
            plateau_scheduler.step(val_mse)

    loss_df = pd.DataFrame(loss_history)
    loss_df.to_csv(os.path.join(output_dir, 'loss_history_' + model_name + '.csv'), index=False)
    print(f'保存损失历史: {os.path.join(output_dir, "loss_history_" + model_name + ".csv")}')

    model.load_state_dict(best_model_state)
    return model, best_metrics

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用设备:", device)
if device.type == 'cuda':
    torch.cuda.set_per_process_memory_fraction(0.8)
torch.cuda.empty_cache()
gc.collect()
print("训练前 - GPU内存分配:", torch.cuda.memory_allocated() / 1e9, "GB")
print("训练前 - GPU内存预留:", torch.cuda.memory_reserved() / 1e9, "GB")

# 加载统计量
try:
    stats = pd.read_csv('stats.csv')
except FileNotFoundError:
    print("错误: stats.csv 文件未找到，请确保文件存在！")
    sys.exit(1)

try:
    index_123 = [int(i[2:]) for i in selected_123]
    index_124 = [int(i[2:]) for i in selected_124]
    max_use_123 = stats[['max']].loc[index_123]
    min_use_123 = stats[['min']].loc[index_123]
    max_use_124 = stats[['max']].loc[index_124]
    min_use_124 = stats[['min']].loc[index_124]
except NameError:
    print("错误: selected_123 或 selected_124 未定义，请确保接口代码已运行！")
    sys.exit(1)

# 创建数据集
try:
    dataset_123 = TimeSeriesDatasetWithNoise(TimeSeriesDataset(data_normalized[selected_123], data_normalized[['信号123']], input_window=input_window_len, pred_window=output_window_len, stride=24), noise_std=0.0)
    dataset_124 = TimeSeriesDatasetWithNoise(TimeSeriesDataset(data_normalized[selected_124], data_normalized[['信号124']], input_window=input_window_len, pred_window=output_window_len, stride=24), noise_std=0.01)
except NameError:
    print("错误: data_normalized 未定义，请确保接口代码已运行！")
    sys.exit(1)

# 数据分割
try:
    train_loader_ts_123, val_loader_ts_123, test_loader_ts_123 = split_time_series(dataset_123)
    train_loader_ts_124, val_loader_ts_124, test_loader_ts_124 = split_time_series(dataset_124)
except NameError:
    print("错误: split_time_series 未定义，请确保接口代码已运行！")
    sys.exit(1)

# 训练信号 123 模型（LSTM+GRU）
print("为信号123训练LSTM+GRU模型...")
input_dim_123 = len(selected_123)  # 5
lstm_hidden_dim_123 = 80
gru_hidden_dim_123 = 32
lstm_layers = 3
gru_layers = 3
output_dim = 1
pred_window = output_window_len

try:
    model_123 = LSTMGRUForecaster(input_dim=input_dim_123, lstm_hidden_dim=lstm_hidden_dim_123, gru_hidden_dim=gru_hidden_dim_123, 
                                 lstm_layers=lstm_layers, gru_layers=gru_layers, output_dim=output_dim, pred_window=pred_window, dropout=0.15).to(device)
    print(f"model_123 初始化，输入维度={input_dim_123}")
    print("model_123 初始化后 - GPU内存分配:", torch.cuda.memory_allocated() / 1e9, "GB")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("model_123 初始化时CUDA内存不足，切换到CPU...")
        device = torch.device('cpu')
        model_123 = LSTMGRUForecaster(input_dim=input_dim_123, lstm_hidden_dim=lstm_hidden_dim_123, gru_hidden_dim=gru_hidden_dim_123, 
                                     lstm_layers=lstm_layers, gru_layers=gru_layers, output_dim=output_dim, pred_window=pred_window, dropout=0.15).to(device)
        dataset_123.tensors = [t.cpu() for t in dataset_123.dataset.tensors]
        dataset_123.dataset.tensors = [t.cpu() for t in dataset_123.dataset.tensors]
        train_loader_ts_123 = DataLoader(dataset_123, batch_size=64, shuffle=True, num_workers=2, pin_memory=False)
        val_loader_ts_123 = DataLoader(dataset_123, batch_size=64, shuffle=False, num_workers=2, pin_memory=False)
    else:
        raise e

model_123, best_metrics_123 = train_ts_model(model_123, train_loader_ts_123, val_loader_ts_123, device, 'lstm_gru_123')

# 预测信号 123
model_123.eval()
with torch.no_grad():
    input_ts_123 = data_normalized[selected_123].iloc[-input_window_len:,:].values.reshape(1, input_window_len, -1)
    print(f"信号123输入形状: {input_ts_123.shape}")
    input_ts_123_tensor = torch.tensor(input_ts_123, dtype=torch.float32).to(device)
    output_123 = model_123(input_ts_123_tensor).squeeze().cpu().numpy()
    output_123 = np.nan_to_num(output_123, nan=0.0, posinf=0.0, neginf=0.0)
    max_val = max_use_123.loc[123].values[0]
    min_val = min_use_123.loc[123].values[0]
    output_123_denorm = output_123 * (max_val - min_val) + min_val
    output_path_123 = os.path.join(output_dir, 'output_lstm_gru_123.pth')
    torch.save({'output': output_123, 'output_denorm': output_123_denorm}, output_path_123)
    print(f"预测信号123（去归一化）: {output_123_denorm}")
    print(f"保存预测: {output_path_123}")

# 清理信号 123 资源
del model_123
gc.collect()
torch.cuda.empty_cache()
print("信号123训练后 - GPU内存分配:", torch.cuda.memory_allocated() / 1e9, "GB")
print("信号123训练后 - GPU内存预留:", torch.cuda.memory_reserved() / 1e9, "GB")

# 训练信号 124 模型（CNN+LSTM）
print("为信号124训练CNN+LSTM模型...")
input_dim_124 = len(selected_124)  # 24
lstm_hidden_dim_124 = 256
num_layers = 3
try:
    model_124 = CNNLSTMForecaster(input_dim=input_dim_124, lstm_hidden_dim=lstm_hidden_dim_124, num_layers=num_layers, 
                                 output_dim=output_dim, pred_window=pred_window, dropout=0.2).to(device)
    print(f"model_124 初始化，输入维度={input_dim_124}")
    print("model_124 初始化后 - GPU内存分配:", torch.cuda.memory_allocated() / 1e9, "GB")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("model_124 初始化时CUDA内存不足，切换到CPU...")
        device = torch.device('cpu')
        model_124 = CNNLSTMForecaster(input_dim=input_dim_124, lstm_hidden_dim=lstm_hidden_dim_124, num_layers=num_layers, 
                                     output_dim=output_dim, pred_window=pred_window, dropout=0.2).to(device)
        dataset_124.tensors = [t.cpu() for t in dataset_124.dataset.tensors]
        dataset_124.dataset.tensors = [t.cpu() for t in dataset_124.dataset.tensors]
        train_loader_ts_124 = DataLoader(dataset_124, batch_size=64, shuffle=True, num_workers=2, pin_memory=False)
        val_loader_ts_124 = DataLoader(dataset_124, batch_size=64, shuffle=False, num_workers=2, pin_memory=False)
    else:
        raise e

model_124, best_metrics_124 = train_ts_model(model_124, train_loader_ts_124, val_loader_ts_124, device, 'cnn_lstm_124')

# 预测信号 124
model_124.eval()
with torch.no_grad():
    input_ts_124 = data_normalized[selected_124].iloc[-input_window_len:,:].values.reshape(1, input_window_len, -1)
    print(f"信号124输入形状: {input_ts_124.shape}")
    input_ts_124_tensor = torch.tensor(input_ts_124, dtype=torch.float32).to(device)
    output_124 = model_124(input_ts_124_tensor).squeeze().cpu().numpy()
    output_124 = np.nan_to_num(output_124, nan=0.0, posinf=0.0, neginf=0.0)
    max_val = max_use_124.loc[124].values[0]
    min_val = min_use_124.loc[124].values[0]
    output_124_denorm = output_124 * (max_val - min_val) + min_val
    output_path_124 = os.path.join(output_dir, 'output_cnn_lstm_124.pth')
    torch.save({'output': output_124, 'output_denorm': output_124_denorm}, output_path_124)
    print(f"预测信号124（去归一化）: {output_124_denorm}")
    print(f"保存预测: {output_path_124}")

# 清理信号 124 资源
del model_124
gc.collect()
torch.cuda.empty_cache()
print("信号124训练后 - GPU内存分配:", torch.cuda.memory_allocated() / 1e9, "GB")
print("信号124训练后 - GPU内存预留:", torch.cuda.memory_reserved() / 1e9, "GB")

# 完成提示
print("训练和预测完成！")
print("信号123（LSTM+GRU）的最佳指标:", best_metrics_123)
print("信号124（CNN+LSTM）的最佳指标:", best_metrics_124)
print("注意：加载模型和预测结果的方法如下：")
print("model_123 = LSTMGRUForecaster(input_dim=5, lstm_hidden_dim=80, gru_hidden_dim=32, lstm_layers=3, gru_layers=3, output_dim=1, pred_window=24, dropout=0.15)")
print("checkpoint = torch.load('" + os.path.join(output_dir, 'best_lstm_gru_123.pth') + "')")
print("model_123.load_state_dict(checkpoint)")
print("prediction = torch.load('" + os.path.join(output_dir, 'output_lstm_gru_123.pth') + "')['output_denorm']")
print("metrics = json.load(open('" + os.path.join(output_dir, 'best_metrics_lstm_gru_123.json') + "'))")
print("model_124 = CNNLSTMForecaster(input_dim=24, lstm_hidden_dim=256, num_layers=3, output_dim=1, pred_window=24, dropout=0.2)")
print("checkpoint = torch.load('" + os.path.join(output_dir, 'best_cnn_lstm_124.pth') + "')")
print("model_124.load_state_dict(checkpoint)")
print("prediction = torch.load('" + os.path.join(output_dir, 'output_cnn_lstm_124.pth') + "')['output_denorm']")
print("metrics = json.load(open('" + os.path.join(output_dir, 'best_metrics_cnn_lstm_124.json') + "'))")

CUDA可用: True
GPU名称: Tesla P100-PCIE-16GB
GPU总内存: 17.071734784 GB
使用设备: cuda
训练前 - GPU内存分配: 0.0 GB
训练前 - GPU内存预留: 0.0 GB
为信号123训练LSTM+GRU模型...
model_123 初始化，输入维度=5
model_123 初始化后 - GPU内存分配: 0.000626176 GB
轮次 1/150, 训练损失: 0.050233, 验证MSE: 0.052317
保存最佳模型: lsM16/best_lstm_gru_123.pth (验证MSE: 0.052317)
保存最佳指标: lsM16/best_metrics_lstm_gru_123.json
轮次 2/150, 训练损失: 0.010625, 验证MSE: 0.005984
保存最佳模型: lsM16/best_lstm_gru_123.pth (验证MSE: 0.005984)
保存最佳指标: lsM16/best_metrics_lstm_gru_123.json
轮次 3/150, 训练损失: 0.002518, 验证MSE: 0.002998
保存最佳模型: lsM16/best_lstm_gru_123.pth (验证MSE: 0.002998)
保存最佳指标: lsM16/best_metrics_lstm_gru_123.json
